# Frente B: descriptores (etapas 6 a 9)

Multitaper, husos, ondas lentas con Woody, CSP, EOG, EMG y HRV. Trabaja sobre `preprocesado_sintetico()` hasta que el Frente A esté listo. Figuras con prefijo `b_`.

In [ ]:
# Entorno común a todos los notebooks: se corre desde codigo/ para que DIR_DATOS
# ("data" o $ESTADIOS_DATOS) apunte al mismo lugar que los scripts.
import os
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

# Si el paquete no está instalado con `pip install -e .`, se usa directamente codigo/src.
# Las dependencias (numpy, scipy, pandas, matplotlib, mne, scikit-learn) sí tienen que
# estar instaladas en el Python de este kernel.
try:
    import estadios  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd() / "src"))
FIGURAS = Path.cwd().parent / "informe" / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt

from estadios.comun.config import Config
from estadios.comun.constantes import NOMBRES_ESTADIOS, N_ESTADIOS, SIN_ANOTAR
from estadios.datos.carga import ids_disponibles

USAR_DATOS_REALES = len(ids_disponibles()) > 0
print("Datos reales" if USAR_DATOS_REALES else "Sin datos descargados: se usan los sintéticos")

In [ ]:
def guardar(fig, nombre):
    """Guarda la figura en informe/figuras/ para incluirla con \\includegraphics."""
    fig.savefig(FIGURAS / f"{nombre}.pdf", bbox_inches="tight")

In [ ]:
from estadios.comun.sintetico import preprocesado_sintetico

cfg = Config().descriptores
prep = preprocesado_sintetico()
registro = prep.registro

## 6. Espectro multitaper vs. Welch

In [ ]:
from estadios.comun.constantes import EEG_C4
from estadios.descriptores import espectro
from estadios.preprocesamiento.segmentacion import segmentar

epocas = segmentar(registro.senales[EEG_C4], registro.fs, registro.n_epocas)
f_mt, psd_mt = espectro.psd_multitaper(epocas, registro.fs, cfg.multitaper_nw, cfg.multitaper_k)
f_w, psd_w = espectro.psd_welch(epocas, registro.fs)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(f_w, psd_w[0], label="Welch", alpha=0.8)
ax.semilogy(f_mt, psd_mt[0], label="Multitaper", alpha=0.8)
ax.set(xlabel="Frecuencia (Hz)", ylabel="PSD (µV²/Hz)", xlim=(0, 30), title="Primera época, " + EEG_C4)
ax.legend()
guardar(fig, "b_multitaper_vs_welch")

## 7. Husos, ondas lentas y promediado de Woody

In [ ]:
from estadios.descriptores import husos, ondas_lentas, woody
# TODO(Frente B): ejemplo de huso detectado y plantilla de Woody vs. promedio ordinario.

## 8. CSP

In [ ]:
from estadios.descriptores import csp
# TODO(Frente B): pesos de cada canal para el par N1-REM.

## 9. EOG, EMG y HRV

In [ ]:
from estadios.descriptores.ensamblado import describir_registro

tabla = describir_registro(prep, cfg)
print(tabla.matriz.shape)
tabla.nombres